# Merge Chunked Inference Results (v2)

**Input:** 13 `results_chunk_XX.csv` files from parallel Colab workers

**Output:** `merged_results_v2.csv` with SHA-512 checksum

**Steps:**
1. Log environment for reproducibility
2. Verify SHA-512 of each uploaded chunk result
3. Merge all 13 chunks into one CSV
4. Print and record SHA-512 of merged output

## 0. Environment Log

In [1]:
# ============================================================
# ENVIRONMENT LOG — Reproducibility
# ============================================================
import sys, platform, datetime, subprocess, shutil, json

env_log = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {},
}

if shutil.which("nvidia-smi"):
    env_log["gpu"] = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip()

if shutil.which("nvcc"):
    env_log["cuda"] = subprocess.check_output(
        ["nvcc", "--version"], text=True).strip().split("\n")[-1]

try:
    import psutil
    env_log["ram_gb"] = round(psutil.virtual_memory().total / 1024**3, 1)
except ImportError:
    pass

for pkg in ["pip", "pandas", "numpy", "hashlib"]:
    try:
        m = __import__(pkg)
        env_log["packages"][pkg] = getattr(m, "__version__", "installed")
    except ImportError:
        env_log["packages"][pkg] = "NOT INSTALLED"

ENV_FILE = "environment_log_merge.json"
with open(ENV_FILE, "w") as f:
    json.dump(env_log, f, indent=2)

print(json.dumps(env_log, indent=2))
print(f"\nSaved to {ENV_FILE}")

{
  "timestamp": "2026-03-16T17:07:18.130462+00:00",
  "python": "3.12.12",
  "platform": "Linux-6.6.113+-x86_64-with-glibc2.35",
  "packages": {
    "pip": "24.1.2",
    "pandas": "2.2.2",
    "numpy": "2.0.2",
    "hashlib": "installed"
  },
  "ram_gb": 12.7
}

Saved to environment_log_merge.json


## 1. Upload & Verify Chunk Results

Upload all 13 `results_chunk_XX.csv` files to Colab, then verify each SHA-512.

In [2]:
import pandas as pd
import hashlib
import os

# ============================================================
# EXPECTED SHA-512 CHECKSUMS (from worker notebook outputs)
# ============================================================
EXPECTED_HASHES = {
    1:  "0332995d4e1a2b4e2d4ccbbf118453e7b21f17c4a879b2cde6fef4bec2ecfa60e69b022077c34e72c2409a4ba19fe2cbd4e9870dd24ce05e9a664945349fda95",
    2:  "824f320d8bd462276027e3bd7a3edb46fa26eebd3851df778393250c9ff8bf03fab3794754609e1a9a7c65563a39cde1b371f28ffdba405b2d2787153ce1c74e",
    3:  "1bd400018b5a9da48207cb455552d45e74fe72d7d22d6ee76f040e797af6dc0ac5bc549a69ffcf6ad6a784040ab5246482f42792d7ad3dbda432a291b6a2d6ed",
    4:  "f43a2b118b1ebab79895f62830c8f2dfb04da375d7d749935bea6df548acdb37aa2d17aff55eedef25296fb612006d60a4f3c964ed4a5d4e50db9fa9377e9e47",
    5:  "1ea89c249a74993873972d430dabad29389032afcccd777f3be30cd1066f2cde6ae9ce025d502999ec417b459e16c55c2ea9b35489e42cb771185f66e22b680b",
    6:  "5aa63db73545b0db0887cb24c2308ed339a4d044a3dfbcf744fdc70d4634bbffc009c734cfbc202247c10641390c814cd4a15765e05129cd0ff1504ae1911b5a",
    7:  "b568ecbe0eae59c6769546f5fa552a7eba50a65e36cfc0d2523b67719ea7a3d14cf77cc34d6ec4c72a114d70ad3a4aca3b8fdd19345c1ad733726a5a594975bd",
    8:  "7aac6062d6b4e9dd1709fec29bec32df7e6320d0fca79572295df65d9850f968fe20e80e718693483446d11e5de672a16752213d4448a50f9101c0db6a9bb6e3",
    9:  "3ae448688bb83cff58c7bd89208a5102758a4bb1c931aa6520a4f5da1431a97123b102fb238ce2da7a6924d6dbfd4e489e4924615fa2d29f75f4d87e106e5cad",
    10: "e8fb9e58a4f39ddef3ea927e6ee4ca21f194bc5c708c272241bae80ca053d69d3a3d4928a883e2db95b59bb485d4e3cc6f89d2bd1ab1bd286cdc6c392ff77292",
    11: "164570621ae7160e0693a3149758bfb06c1d98e84bc7f687f397c8e0f8531e1480ce0cac05597e1de9c96e722b7c2f6ea7944f085816edd3d84c5c2dcd006d17",
    12: "f88abc0fa191f42c59206189bce9d3ef4c8be82a2e7d1ab0a143d9c1f31e535eef505365ebd29f7052e403184a2f76860db1da53a1ccb9334a66560de5979392",
    13: "86d59a8d15c2cbaa0ef8cb793b2f2a3fa6beb019ee2927ee9fd0e1409f5d21503ddc1733f7271dfa44c0f6fb6a0f80f708efca6376e75f4a324b56b16466c475"
}

NUM_CHUNKS = 13


def sha512_file(path):
    """Compute SHA-512 of a file."""
    h = hashlib.sha512()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8192), b""):
            h.update(block)
    return h.hexdigest()


# ============================================================
# VERIFY EACH CHUNK
# ============================================================
print("=" * 65)
print("INTEGRITY CHECK — 13 chunk result files")
print("=" * 65)

all_ok = True
chunk_files = {}

for chunk_id in range(1, NUM_CHUNKS + 1):
    filename = f"results_chunk_{chunk_id:02d}.csv"

    if not os.path.exists(filename):
        print(f"  Chunk {chunk_id:2d}: MISSING — {filename} not found!")
        all_ok = False
        continue

    actual = sha512_file(filename)
    expected = EXPECTED_HASHES[chunk_id]

    if actual == expected:
        rows = len(pd.read_csv(filename))
        print(f"  Chunk {chunk_id:2d}: PASS  ({rows:,} rows)")
        chunk_files[chunk_id] = filename
    else:
        print(f"  Chunk {chunk_id:2d}: FAIL")
        print(f"    Expected: {expected}")
        print(f"    Got:      {actual}")
        all_ok = False

print("=" * 65)
if all_ok and len(chunk_files) == NUM_CHUNKS:
    print(f"All {NUM_CHUNKS} chunks verified successfully.")
else:
    missing = NUM_CHUNKS - len(chunk_files)
    print(f"VERIFICATION FAILED: {missing} chunk(s) missing or corrupted.")
    raise ValueError("Cannot merge — integrity check failed. Re-upload the failed chunks.")

INTEGRITY CHECK — 13 chunk result files
  Chunk  1: PASS  (2,787 rows)
  Chunk  2: PASS  (2,787 rows)
  Chunk  3: PASS  (2,787 rows)
  Chunk  4: PASS  (2,787 rows)
  Chunk  5: PASS  (2,787 rows)
  Chunk  6: PASS  (2,787 rows)
  Chunk  7: PASS  (2,787 rows)
  Chunk  8: PASS  (2,787 rows)
  Chunk  9: PASS  (2,787 rows)
  Chunk 10: PASS  (2,787 rows)
  Chunk 11: PASS  (2,787 rows)
  Chunk 12: PASS  (2,787 rows)
  Chunk 13: PASS  (2,809 rows)
All 13 chunks verified successfully.


## 2. Merge All Chunks

In [3]:
# ============================================================
# MERGE 13 CHUNKS INTO ONE CSV
# ============================================================
dfs = []
for chunk_id in range(1, NUM_CHUNKS + 1):
    df = pd.read_csv(chunk_files[chunk_id])
    df["chunk_id"] = chunk_id
    dfs.append(df)
    print(f"  Chunk {chunk_id:2d}: {len(df):,} rows")

df_merged = pd.concat(dfs, ignore_index=True)

OUTPUT_FILE = "merged_results_v2.csv"
df_merged.to_csv(OUTPUT_FILE, index=False)

print(f"\nMerged: {len(df_merged):,} total rows")
print(f"Saved:  {OUTPUT_FILE}")

  Chunk  1: 2,787 rows
  Chunk  2: 2,787 rows
  Chunk  3: 2,787 rows
  Chunk  4: 2,787 rows
  Chunk  5: 2,787 rows
  Chunk  6: 2,787 rows
  Chunk  7: 2,787 rows
  Chunk  8: 2,787 rows
  Chunk  9: 2,787 rows
  Chunk 10: 2,787 rows
  Chunk 11: 2,787 rows
  Chunk 12: 2,787 rows
  Chunk 13: 2,809 rows

Merged: 36,253 total rows
Saved:  merged_results_v2.csv


## 3. Merged Results Summary

In [4]:
# ============================================================
# SUMMARY STATISTICS
# ============================================================
VALID_LABELS = {"junior", "mid", "senior"}

valid_mask = df_merged["predicted_level"].isin(VALID_LABELS)
valid = df_merged[valid_mask]
unknown_rate = 1 - valid_mask.mean()

acc = (valid["predicted_level"] == valid["exp_level_3"]).mean()

print("=" * 50)
print("MERGED RESULTS SUMMARY")
print("=" * 50)
print(f"  Total rows:      {len(df_merged):,}")
print(f"  Valid preds:     {len(valid):,} ({valid_mask.mean():.1%})")
print(f"  Unknown rate:    {unknown_rate:.4%}")
print(f"  Overall accuracy:{acc:.4f}")
print()
print("Predicted label distribution:")
print(df_merged["predicted_level"].value_counts().to_string())
print()
print("Ground truth distribution:")
print(df_merged["exp_level_3"].value_counts().to_string())
print()

# Per-chunk accuracy
print("Per-chunk accuracy:")
for chunk_id in range(1, NUM_CHUNKS + 1):
    chunk = df_merged[df_merged["chunk_id"] == chunk_id]
    v = chunk[chunk["predicted_level"].isin(VALID_LABELS)]
    a = (v["predicted_level"] == v["exp_level_3"]).mean()
    print(f"  Chunk {chunk_id:2d}: {a:.4f} ({len(chunk):,} rows)")

MERGED RESULTS SUMMARY
  Total rows:      36,253
  Valid preds:     36,252 (100.0%)
  Unknown rate:    0.0028%
  Overall accuracy:0.5492

Predicted label distribution:
predicted_level
junior     14267
mid        13275
senior      8710
unknown        1

Ground truth distribution:
exp_level_3
junior    19507
mid       15104
senior     1642

Per-chunk accuracy:
  Chunk  1: 0.5310 (2,787 rows)
  Chunk  2: 0.5450 (2,787 rows)
  Chunk  3: 0.5472 (2,787 rows)
  Chunk  4: 0.5536 (2,787 rows)
  Chunk  5: 0.5314 (2,787 rows)
  Chunk  6: 0.5633 (2,787 rows)
  Chunk  7: 0.5603 (2,787 rows)
  Chunk  8: 0.5579 (2,787 rows)
  Chunk  9: 0.5716 (2,787 rows)
  Chunk 10: 0.5673 (2,787 rows)
  Chunk 11: 0.5511 (2,787 rows)
  Chunk 12: 0.5088 (2,787 rows)
  Chunk 13: 0.5507 (2,809 rows)


## 4. Output SHA-512

In [5]:
# ============================================================
# OUTPUT FILE SHA-512 CHECKSUM
# ============================================================
merged_hash = sha512_file(OUTPUT_FILE)

print(f"Output file: {OUTPUT_FILE}")
print(f"Rows:        {len(df_merged):,}")
print(f"SHA-512:     {merged_hash}")

# Save hash to a sidecar file for the repo
with open("merged_results_v2.sha512", "w") as f:
    f.write(f"{merged_hash}  {OUTPUT_FILE}\n")
print(f"\nHash saved to merged_results_v2.sha512")

Output file: merged_results_v2.csv
Rows:        36,253
SHA-512:     d2d47f992e0ae0a9d56f8499c2246d3dbb2e20b4cf759bd5891212c9cf90ffd0b13bef43aa6708faaa88e87473a31d3326b8e3d8cba760cb0452909fddc57ab4

Hash saved to merged_results_v2.sha512
